In [5]:
pip install groq


   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   -------------------------- ------------- 2/3 [groq]
   ---------------------------------------- 3/3 [groq]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [6]:
import os
from groq import Groq

# Initialize the client (ensure your API key is in your environment variables)
client = Groq(api_key="your key")

def generate_text(prompt):
    completion = client.chat.completions.create(
        # Llama 3.1 8B is a great "instant" free-tier model
        model="llama-3.1-8b-instant", 
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=10,
        temperature=0.7,
    )
    return completion.choices[0].message.content.strip()

In [7]:
prompt = "Once upon a time"

In [8]:
generated_text = generate_text(prompt)
print(prompt, generated_text)

Once upon a time in a far-off kingdom, where magic filled the


## Customizing the Output

In [11]:
def generate_text(prompt,max_tokens, temperature):
    completion = client.chat.completions.create(
        # Llama 3.1 8B is a great "instant" free-tier model
        model="llama-3.1-8b-instant", 
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return completion.choices[0].message.content.strip()

In [12]:
generated_text = generate_text(prompt, 50, 0)
print(prompt, generated_text)

Once upon a time ...in a far-off kingdom, hidden behind a veil of mist and legend, there existed a magical realm where wonder and enchantment reigned supreme.


In [13]:
generated_text = generate_text(prompt, 50, 1)
print(prompt, generated_text)

Once upon a time in a land far, far away...


## Summarising Text

In [14]:
def text_summarizer(prompt):
    response = client.chat.completions.create(
        # Llama 3.1 8B is perfect for fast, small tasks like keyword extraction
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system", 
                "content": "You will be provided with a block of text, and your task is to extract a list of keywords from it."
            },
            {
                "role": "user", 
                "content": "A flying saucer seen by a guest house, a 7ft alien-like figure coming out of a hedge and a \"cigar-shaped\" UFO near a school yard..." # (Text truncated for brevity)
            },
            {
                "role": "assistant", 
                "content": "flying saucer, guest house, 7ft alien-like figure, hedge, cigar-shaped UFO, school yard..." # (Text truncated for brevity)
            },
            {
                "role": "user", 
                "content": "Each April, in the village of Maeliya in northwest Sri Lanka..." # (Text truncated for brevity)
            },
            {
                "role": "assistant", 
                "content": "April, Maeliya, northwest Sri Lanka, Pinchal Weldurelage Siriwardene..." # (Text truncated for brevity)
            },
            {
                "role": "user", 
                "content": prompt
            }
        ],
        temperature=0.5,
        max_tokens=256
    )
    
    # The response structure is the same as modern OpenAI
    return response.choices[0].message.content.strip()

In [15]:
prompt = "Master Reef Guide Kirsty Whitman didn't need to tell me twice. Peering down through my snorkel mask in the direction of her pointed finger, I spotted a huge male manta ray trailing a female in perfect sync – an effort to impress a potential mate, exactly as Whitman had described during her animated presentation the previous evening. Having some knowledge of what was unfolding before my eyes on our snorkelling safari made the encounter even more magical as I kicked against the current to admire this intimate undersea ballet for a few precious seconds more."
print(prompt)

Master Reef Guide Kirsty Whitman didn't need to tell me twice. Peering down through my snorkel mask in the direction of her pointed finger, I spotted a huge male manta ray trailing a female in perfect sync – an effort to impress a potential mate, exactly as Whitman had described during her animated presentation the previous evening. Having some knowledge of what was unfolding before my eyes on our snorkelling safari made the encounter even more magical as I kicked against the current to admire this intimate undersea ballet for a few precious seconds more.


In [16]:
text_summarizer(prompt)

'manta ray, snorkel, snorkelling safari, mate, presentation, current, ballet, Kirsty Whitman, undersea.'

## Poetic Chatbot

In [19]:
def poetic_chatbot(prompt):
    response = client.chat.completions.create(
        model = "llama-3.1-8b-instant",
        messages = [
            {
                "role": "system",
                "content": "You are a poetic chatbot."
            },
            {
                "role": "user",
                "content": "When was Google founded?"
            },
            {
                "role": "assistant",
                "content": "In the late '90s, a spark did ignite, Google emerged, a radiant light. By Larry and Sergey, in '98, it was born, a search engine new, on the web it was sworn."
            },
            {
                "role": "user",
                "content": "Which country has the youngest president?"
            },
            {
                "role": "assistant",
                "content": "Ah, the pursuit of youth in politics, a theme we explore. In Austria, Sebastian Kurz did implore, at the age of 31, his journey did begin, leading with vigor, in a world filled with din."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature = 1,
        max_tokens=256
    )
    return response.choices[0].message.content.strip()

In [20]:
prompt = "When was cheese first made?"
poetic_chatbot(prompt)

"A tale of dairy's ancient delight, a story of cheese in the pale moon's light. The dates obscure, but legend does tell, the birth of cheese, perhaps, around 8000 years ago, when nomads did dwell. In the land of the Caucasus, some claim, the art of cheese-making first did gain, its name in the annals, of culinary fame."

In [21]:
prompt = "What is the next course to be uploaded to 365DataScience?"
poetic_chatbot(prompt)

"A quest for knowledge, ever so grand. On 365 Data Science, new paths unfold, a world of data, to be told. Though I'm unaware of the exact course to come, a realm of discovery, will forever be hum.\n\nHowever, I can suggest some popular courses on the platform. If you are interested, I can provide information on those."

## Langchain

In [22]:
from langchain.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.memory import ConversationBufferMemory
from langchain.llms import OpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.chat_models import ChatOpenAI

In [23]:
url = "https://365datascience.com/upcoming-courses"

In [24]:
loader = WebBaseLoader(url)

In [25]:
raw_documents = loader.load()

In [26]:
text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(raw_documents)

In [27]:
embeddings = OpenAIEmbeddings(openai_api_key = api_key)

NameError: name 'api_key' is not defined

In [22]:
vectorstore = FAISS.from_documents(documents, embeddings)

In [23]:
memory = ConversationBufferMemory(memory_key = "chat_history", return_messages=True)

In [24]:
qa = ConversationalRetrievalChain.from_llm(ChatOpenAI(openai_api_key=api_key, 
                                                  model="gpt-3.5-turbo", 
                                                  temperature=0), 
                                           vectorstore.as_retriever(), 
                                           memory=memory)

In [25]:
query = "What is the next course to be uploaded on the 365DataScience platform?"

In [26]:
result = qa({"question": query})

In [27]:
result["answer"]

'The next course to be uploaded on the 365DataScience platform is "LLM Engineering in Practice with Streamlit and OpenAI" with Petar Petrov. It is a hands-on course that explores the world of AI solutions, focusing on designing, developing, and deploying an interview simulator project using Streamlit.'